In [0]:
import re, json, requests

spark.sql('USE CATALOG delivery_cata')

DATABRICKS_HOST  = spark.conf.get('spark.databricks.workspaceUrl')
DATABRICKS_TOKEN = (dbutils.notebook.entry_point
                          .getDbutils().notebook().getContext()
                          .apiToken().get())
MODEL_ENDPOINT   = 'databricks-gpt-oss-20b'
API_URL          = f'https://{DATABRICKS_HOST}/serving-endpoints/{MODEL_ENDPOINT}/invocations'

#Parse model response 
def _parse_content(raw) -> str:
    """
    GPT OSS 20B returns content in 3 formats:
    1. str  → return directly
    2. list[{type:text, text:...}]  → return only the text blocks
    3. list with both reasoning + text blocks → skip reasoning,
       return only the text block (the actual final answer).
    """
    if isinstance(raw, str):
        return raw.strip()
    if isinstance(raw, list):
        # Collect only 'text' blocks -- skip 'reasoning' 
        text_parts = [
            block.get('text', '')
            for block in raw
            if isinstance(block, dict) and block.get('type') == 'text'
        ]
        if text_parts:
            return ' '.join(p for p in text_parts if p).strip()
        # Fallback: no explicit text block -- grab any non-reasoning content
        fallback = [
            block.get('text', '') or block.get('content', '')
            for block in raw
            if isinstance(block, dict) and block.get('type') != 'reasoning'
        ]
        return ' '.join(p for p in fallback if p).strip()
    return str(raw).strip()

# Quick connectivity test
resp = requests.post(
    API_URL,
    headers={'Authorization': f'Bearer {DATABRICKS_TOKEN}',
             'Content-Type': 'application/json'},
    json={'messages': [{'role': 'user', 'content': 'Say: READY'}],
          'max_tokens': 20},
    timeout=30
)
if resp.status_code == 200:
    reply = _parse_content(resp.json()['choices'][0]['message']['content'])
    print(f'Model connected — reply: "{reply}"')
else:
    print(f'Error {resp.status_code}: {resp.text[:200]}')


Model connected — reply: ""


In [0]:
SCHEMA_CONTEXT = """
You are a senior SQL analyst for a delivery platform in India.
You write Spark SQL queries against tables in catalog `delivery_cata`.

=============================================================
PRIMARY TABLE — use this for ALL questions (11,985 rows)
=============================================================

delivery_cata.gold_rpt.fact_delivery
  -- One row per delivery. Join to dims for names/details.
  delivery_sk    BIGINT   (surrogate PK, auto-generated — do NOT filter on this)
  delivery_id    INT
  order_id       INT
  driver_sk      INT      FK → dim_driver.driver_sk
  customer_sk    INT      FK → dim_customer.customer_sk
  vehicle_sk     INT      FK → dim_vehicle.vehicle_sk
  area_sk        INT      FK → dim_area.area_sk
  payment_sk     INT      FK → dim_payment.payment_sk
  date_sk        INT      FK → dim_date.date_sk  (format YYYYMMDD e.g. 20231212)
  order_status   STRING   PLACED|CONFIRMED|IN_TRANSIT|DELIVERED|CANCELLED|FAILED
  delivery_status STRING  IN_TRANSIT|DELIVERED|FAILED|CANCELLED
  pickup_location   STRING
  delivery_location STRING
  order_amount   DOUBLE   ← ALWAYS use f.order_amount
  payment_amount DOUBLE   ← ALWAYS use f.payment_amount
  distance_km    DOUBLE
  delivery_time_mins INT
  delivery_charge DOUBLE  ← ALWAYS use f.delivery_charge
  pickup_time    TIMESTAMP
  delivery_time  TIMESTAMP
  order_date     DATE     ← USE THIS for date filtering on order questions
  payment_time   TIMESTAMP

=============================================================
DIMENSION TABLES
=============================================================

delivery_cata.gold_rpt.dim_driver  (26,203 rows)
  driver_sk    INT   PK
  driver_id    INT
  driver_name  STRING   ← use LIKE for name matching
  driver_phone STRING
  driver_status STRING  ACTIVE|INACTIVE
  is_current   BOOLEAN  ← ALWAYS add: AND dr.is_current = true
  effective_start TIMESTAMP
  effective_end   TIMESTAMP

delivery_cata.gold_rpt.dim_customer  (24,915 rows)
  customer_sk    INT   PK
  customer_id    INT
  customer_name  STRING
  customer_email STRING
  customer_phone STRING
  customer_address STRING
  is_current     BOOLEAN  ← ALWAYS add: AND dc.is_current = true

delivery_cata.gold_rpt.dim_area  (26,252 rows)
  area_sk        INT   PK
  area_id        INT
  area_name      STRING
  city           STRING
  pincode        STRING   ← STRING type, always quote: pincode = '302001'
  delivery_charge DOUBLE  ← NOTE: Use fact_delivery.delivery_charge instead.

delivery_cata.gold_rpt.dim_vehicle  (25,926 rows)
  vehicle_sk     INT   PK
  vehicle_id     INT
  vehicle_type   STRING   BIKE|VAN|TRUCK|CAR|SCOOTER
  plate_number   STRING
  capacity_kg    DOUBLE
  vehicle_status STRING

delivery_cata.gold_rpt.dim_payment  (7,644 rows)
  payment_sk     INT   PK
  payment_id     INT
  payment_method STRING   CARD|UPI|CASH|WALLET|NETBANKING
  payment_status STRING   PENDING|COMPLETED|FAILED|REFUNDED
  -- NOTE: payment_amount is NOT in this table. Use fact_delivery.payment_amount.

delivery_cata.gold_rpt.dim_date  (914 dates, range 2023–2025)
  date_sk        INT   PK  format YYYYMMDD
  full_date      DATE
  year           INT
  quarter        INT
  month          INT     1-12
  month_name     STRING  January|February|...
  week_of_year   INT
  day_of_month   INT
  day_of_week    INT     1=Sunday, 7=Saturday
  day_name       STRING  Monday|Tuesday|...
  is_weekend     BOOLEAN

=============================================================
FLAT TABLE — use for simple single-table lookups
=============================================================

delivery_cata.gold_ai.master_delivery_fact  (2,007 rows)
  -- All fields pre-joined. Good for quick driver/area/date lookups.
  delivery_id, order_id,
  driver_id, driver_name, driver_phone, driver_status,
  vehicle_id, vehicle_type, vehicle_plate_number, vehicle_capacity_kg,
  area_id, area_name, city, pincode STRING, delivery_charge,
  customer_id, customer_name, customer_email, customer_phone, customer_address,
  order_amount, order_status, pickup_location, delivery_location,
  order_date DATE, delivery_status, pickup_time, delivery_time, distance_km,
  payment_id, payment_method, payment_status, payment_amount, payment_time

=============================================================
QUERY WRITING RULES — follow exactly
=============================================================

1. TABLE CHOICE:
   - Simple count/filter by driver+date+pincode+payment_method → gold_ai.master_delivery_fact
   - Complex joins, aggregates, trends → gold_rpt.fact_delivery + dims

2. JOINS (gold_rpt):
   JOIN delivery_cata.gold_rpt.dim_driver   dr ON f.driver_sk   = dr.driver_sk   AND dr.is_current = true
   JOIN delivery_cata.gold_rpt.dim_customer dc ON f.customer_sk = dc.customer_sk AND dc.is_current = true
   JOIN delivery_cata.gold_rpt.dim_area     a  ON f.area_sk     = a.area_sk
   JOIN delivery_cata.gold_rpt.dim_vehicle  v  ON f.vehicle_sk  = v.vehicle_sk
   JOIN delivery_cata.gold_rpt.dim_payment  p  ON f.payment_sk  = p.payment_sk
   JOIN delivery_cata.gold_rpt.dim_date     dt ON f.date_sk     = dt.date_sk

3. AMOUNT COLUMNS:
   - payment_amount, order_amount, and delivery_charge are in the FACT table (f).
   - NEVER use p.payment_amount or a.delivery_charge. Use f.payment_amount, f.order_amount, f.delivery_charge.

4. DRIVER NAME: ALWAYS use LIKE (names repeat across many drivers):
   LOWER(driver_name) LIKE LOWER('%Rohit%')

4. DATE FILTERING:
   - "12 Dec 2023"    → order_date = '2023-12-12'   (on master_delivery_fact)
   - "12 Dec 2023"    → dt.full_date = '2023-12-12' (on gold_rpt with dim_date)
   - "December 2023"  → dt.year = 2023 AND dt.month = 12
   - "this year"      → dt.year = YEAR(CURRENT_DATE())
   - "last month"     → dt.year = YEAR(ADD_MONTHS(CURRENT_DATE(),-1))
                        AND dt.month = MONTH(ADD_MONTHS(CURRENT_DATE(),-1))
   - Never use DATE(pickup_time) for filtering — use order_date

5. PINCODE: STRING type — always quote: pincode = '302001'

6. OUTPUT FORMAT: Return ONLY the SQL inside ```sql ... ``` — nothing else.

=============================================================
EXAMPLES
=============================================================

Q: How many orders were assigned to driver Rohit on 12 Dec 2023 in pincode 160017?
A:
```sql
SELECT COUNT(*) AS total_orders
FROM delivery_cata.gold_ai.master_delivery_fact
WHERE LOWER(driver_name) LIKE LOWER('%Rohit%')
  AND order_date = '2023-12-12'
  AND pincode = '160017'
```

Q: Which driver delivered the most orders this year?
A:
```sql
SELECT dr.driver_name, COUNT(*) AS total_delivered
FROM delivery_cata.gold_rpt.fact_delivery f
JOIN delivery_cata.gold_rpt.dim_driver dr ON f.driver_sk = dr.driver_sk AND dr.is_current = true
JOIN delivery_cata.gold_rpt.dim_date   dt ON f.date_sk   = dt.date_sk
WHERE dt.year = YEAR(CURRENT_DATE())
  AND f.delivery_status = 'DELIVERED'
GROUP BY dr.driver_name
ORDER BY total_delivered DESC
LIMIT 1
```

Q: What is total UPI revenue in pincode 302001?
A:
```sql
SELECT ROUND(SUM(payment_amount), 2) AS total_upi_revenue
FROM delivery_cata.gold_ai.master_delivery_fact
WHERE payment_method = 'UPI'
  AND pincode = '302001'
```
"""

print(f'Schema context loaded: {len(SCHEMA_CONTEXT)} chars')


Schema context loaded: 7171 chars


In [0]:
HISTORY_TABLE = 'delivery_cata.gold_ai.chat_history'

def init_history_table():
    """Create the history table if it doesn't exist."""
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {HISTORY_TABLE} (
            timestamp TIMESTAMP,
            question STRING,
            answer STRING
        )
        USING DELTA
    """)

def load_history(limit=5):
    """Load the last N turns from the history table."""
    try:
        df = spark.sql(f"SELECT question, answer FROM {HISTORY_TABLE} ORDER BY timestamp DESC LIMIT {limit}")
        # Convert to list of dicts and reverse to get chronological order
        history = [{"q": r.question, "a": r.answer} for r in df.collect()]
        return history[::-1]
    except Exception as e:
        print(f"Note: Could not load history ({e})")
        return []

def save_history(question, answer):
    """Save a new chat turn to the history table."""
    from datetime import datetime
    try:
        new_row = spark.createDataFrame([(datetime.now(), question, answer)], 
                                       ["timestamp", "question", "answer"])
        new_row.write.format("delta").mode("append").saveAsTable(HISTORY_TABLE)
    except Exception as e:
        print(f"Note: Could not save history ({e})")

def clear_history_table():
    """Clear all records from the history table."""
    try:
        spark.sql(f"TRUNCATE TABLE {HISTORY_TABLE}")
    except Exception as e:
        print(f"Note: Could not clear history ({e})")

init_history_table()

In [0]:
# NL→SQL→NL Engine
import json, re

# LLM caller
def call_llm(messages: list, max_tokens: int = 512,
             temperature: float = 0.0) -> str:
    resp = requests.post(
        API_URL,
        headers={'Authorization': f'Bearer {DATABRICKS_TOKEN}',
                 'Content-Type': 'application/json'},
        json={'messages':    messages,
              'max_tokens':  max_tokens,
              'temperature': temperature},
        timeout=90
    )
    resp.raise_for_status()
    raw = resp.json()['choices'][0]['message']['content']
    return _parse_content(raw)


#SQL extractor — handles GPT OSS 20B reasoning output
def extract_sql(raw: str) -> str:
    """
    GPT OSS 20B often returns a reasoning block like:
    'SELECT ... AS total_spent. Group by ... Use correct table names. Use ...'
    
    We must extract ONLY the SQL SELECT or WITH statement — nothing after the first
    non-SQL sentence (anything ending with '.' that isn't SQL).
    """
    # try ```sql ... ``` fence first — cleanest format
    m = re.search(r'```(?:sql)?\s*([\s\S]+?)```', raw, re.IGNORECASE)
    if m:
        sql = m.group(1).strip()
        if re.match(r'^(SELECT|WITH)\b', sql, re.IGNORECASE):
            return sql

    # find SELECT/WITH and take only SQL lines — stop at non-SQL prose
    lines = raw.splitlines()
    sql_lines = []
    in_sql    = False

    SQL_KEYWORDS = re.compile(
        r'^\s*(SELECT|FROM|WHERE|JOIN|LEFT|RIGHT|INNER|OUTER|GROUP|ORDER|'
        r'HAVING|LIMIT|WITH|ON|AND|OR|UNION|CASE|WHEN|THEN|ELSE|END|'
        r'COUNT|SUM|AVG|MAX|MIN|ROUND|CAST|DATE|YEAR|MONTH|DAY|'
        r'AS|BY|IS|NOT|NULL|TRUE|FALSE|IN|LIKE|BETWEEN|DESC|ASC)',
        re.IGNORECASE
    )
    # Pattern that identifies prose (starts with "Use ", ends with ".", etc.)
    PROSE = re.compile(
        r'^(Use |Note |Make sure|Always |Never |Do |Don\'t |This |The |A |An )',
        re.IGNORECASE
    )

    for line in lines:
        stripped = line.strip()
        if not stripped:
            if in_sql:
                sql_lines.append('')
            continue

        # Found SELECT or WITH — start collecting
        if re.match(r'(SELECT|WITH)\b', stripped, re.IGNORECASE):
            in_sql = True

        if in_sql:
            # Stop if it looks like prose instructions
            if PROSE.match(stripped):
                break
            # Stop if line ends with '.' and is not SQL
            if stripped.endswith('.') and not SQL_KEYWORDS.match(stripped):
                break
            sql_lines.append(line)

    if sql_lines:
        sql = '\n'.join(sql_lines).strip()
        # Remove any trailing period (GPT OSS 20B sometimes adds these)
        sql = sql.rstrip('. \t\n')
        if re.match(r'^(SELECT|WITH)\b', sql, re.IGNORECASE):
            return sql

    #last resort — regex grab SELECT/WITH to end of string
    m = re.search(r'((?:SELECT|WITH)\b[\s\S]+)', raw, re.IGNORECASE)
    if m:
        candidate = m.group(1).strip()
        # Cut off at first prose sentence
        cut = re.search(
            r'\n(Use |Note |Make sure|Always |Never |---)', 
            candidate, re.IGNORECASE
        )
        if cut:
            candidate = candidate[:cut.start()].strip()
        return candidate.rstrip('.')

    return ''


#SQL validator
def validate_sql(sql: str) -> bool:
    if not sql or len(sql) < 10:
        return False
    # Added WITH here as well to pass the security check
    if not re.match(r'^\s*(SELECT|WITH)\b', sql, re.IGNORECASE):
        return False
    blocked = re.compile(
        r'\b(INSERT|UPDATE|DELETE|DROP|CREATE|ALTER|TRUNCATE|MERGE)\b',
        re.IGNORECASE)
    return not bool(blocked.search(sql))


#  NL → SQl
def nl_to_sql(question: str, history: list = []) -> str:
    # Build context from history
    history_context = ""
    if history:
        history_context = "\nRecent Conversation History:\n"
        for turn in history:
            history_context += f"User: {turn['q']}\nAI: {turn['a']}\n"

    messages = [
        {
            'role':    'system',
            'content': SCHEMA_CONTEXT + history_context
        },
        {
            'role':    'user',
            'content': (
                f'Write a Spark SQL query to answer this question:\n\n'
                f'{question}\n\n'
                f'Output ONLY a SQL code block like this:\n'
                f'```sql\n'
                f'SELECT ...\n'
                f'FROM ...\n'
                f'```\n'
                f'Do not write anything outside the code block.'
            )
        }
    ]
    raw = call_llm(messages, max_tokens=600, temperature=0.0)
    print(f'[RAW MODEL OUTPUT]\n{raw}\n{"─"*50}')
    sql = extract_sql(raw)
    print(f'[EXTRACTED SQL]\n{sql}\n{"─"*50}')
    return sql


# Result → NL 
def result_to_nl(question: str, result_json: str, row_count: int, history: list = []) -> str:
    if row_count == 0:
        data_desc = (
            'The query returned 0 rows — no matching records found '
            'for those filters.'
        )
    else:
        data_desc = f'Query returned {row_count} row(s):\n{result_json}'

    # Build context from history
    history_context = ""
    if history:
        history_context = "\nRecent Conversation History:\n"
        for turn in history:
            history_context += f"User: {turn['q']}\nAI: {turn['a']}\n"

    messages = [
        {
            'role':    'system',
            'content': (
                'You are a data analyst. Answer the delivery business question '
                'in 1-2 complete sentences using ONLY the numbers and names '
                'from the data provided.\n\n'
                'RULES:\n'
                '- Never say "Based on the data", "According to results", '
                '"The query shows"\n'
                '- Never mention SQL, tables, JSON, or databases\n'
                '- Always include the exact number from the data\n'
                '- Always include key context (driver name, date, pincode) '
                'from the question\n'
                '- If 0 rows: say clearly that no records were found\n'
                '- Complete your sentence — never stop mid-word\n\n'
                'EXAMPLE:\n'
                'Question: How many orders for Rohit in pincode 160017?\n'
                'Data: [{"total_orders": 6}]\n'
                'Answer: 6 orders were assigned to driver Rohit in '
                'pincode 160017.'
                + history_context
            )
        },
        {
            'role':    'user',
            'content': (
                f'Question: {question}\n\n'
                f'Data: {data_desc}\n\n'
                f'Answer:'
            )
        }
    ]
    return call_llm(messages, max_tokens=300, temperature=0.2)


#Main agent
def ask(question: str, history: list = [], max_rows: int = 50) -> str:
    """
    Full NL → SQL → execute → NL pipeline.
    Input:  plain English question
    Output: plain English answer
    """
    try:
        # generate SQL
        sql = nl_to_sql(question, history)

        if not validate_sql(sql):
            print(f'[WARN] Could not extract valid SQL.\nRaw was above.')
            return (
                'Sorry, the model could not generate a valid SQL query. '
                'Try rephrasing your question more specifically.'
            )

        # run SQL
        df        = spark.sql(sql).limit(max_rows)
        rows      = df.collect()
        row_count = len(rows)
        cols      = df.columns

        # Convert Row → dict → JSON string (no type errors)
        rows_as_dicts = [
            {c: (str(row[c]) if row[c] is not None else None) for c in cols}
            for row in rows
        ]
        result_json = json.dumps(rows_as_dicts[:20])
        print(f'[RESULT] {row_count} rows | {result_json[:200]}')

        # generate NL answer
        return result_to_nl(question, result_json, row_count, history)

    except Exception as e:
        return f'Error: {str(e)}'


print('Engine ready — call ask("your question")')


Engine ready — call ask("your question")


In [0]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Chat history list (pre-loaded from persistent storage)
chat_history = load_history()

q_input = widgets.Textarea(
    placeholder='Ask anything about your deliveries...',
    layout=widgets.Layout(width='100%', height='60px')
)
ask_btn = widgets.Button(
    description='Send ➤',
    button_style='primary',
    layout=widgets.Layout(width='90px', height='36px')
)
clear_btn = widgets.Button(
    description='Clear Chat',
    layout=widgets.Layout(width='90px', height='36px')
)
out = widgets.Output()

def render_chat():
    with out:
        clear_output(wait=True)
        html = '<div style="max-height:500px;overflow-y:auto;padding:10px;">'
        for item in chat_history:
            # User bubble
            html += f'''
            <div style="display:flex;justify-content:flex-end;margin:8px 0;">
              <div style="background:#1a56db;color:white;padding:10px 14px;
                          border-radius:16px 16px 4px 16px;max-width:75%;
                          font-size:14px;">{item["q"]}</div>
            </div>'''
            # Bot bubble
            color = "#fef2f2" if item["a"].startswith("Error") else "#f0fdf4"
            border = "#fca5a5" if item["a"].startswith("Error") else "#16a34a"
            text_color = "#b91c1c" if item["a"].startswith("Error") else "#15803d"
            html += f'''
            <div style="display:flex;justify-content:flex-start;margin:8px 0;">
              <div style="background:{color};border:1px solid {border};color:{text_color};
                          padding:10px 14px;border-radius:16px 16px 16px 4px;
                          max-width:75%;font-size:14px;">{item["a"]}</div>
            </div>'''
        html += '</div>'
        display(HTML(html))

def on_ask(btn):
    question = q_input.value.strip()
    if not question:
        return
    q_input.value = ''
    with out:
        clear_output(wait=True)
        display(HTML('<div style="color:#6b7280;font-size:13px;padding:10px;">⏳ Thinking...</div>'))
    
    # Pass current memory (chat_history) to get context
    answer = ask(question, history=chat_history)
    
    # Save both locally and to persistent Delta table
    save_history(question, answer)
    chat_history.append({"q": question, "a": answer})
    
    # Keep local memory at last 5 turns to match prompt context depth
    if len(chat_history) > 5:
        chat_history.pop(0)
        
    render_chat()

def on_clear(btn):
    chat_history.clear()
    with out:
        clear_output()

ask_btn.on_click(on_ask)
clear_btn.on_click(on_clear)

# Header
display(HTML('''
<div style="background:linear-gradient(135deg,#1e3a8a,#1a56db);color:white;
            padding:20px 24px;border-radius:12px;margin-bottom:16px;">
  <div style="font-size:22px;font-weight:700;">🚚 Delivery Analytics AI</div>
  <div style="font-size:13px;opacity:0.85;margin-top:4px;">
    Ask anything in plain English — powered by GPT OSS 20B
  </div>
</div>
'''))

display(HTML('<div style="font-size:12px;font-weight:600;color:#6b7280;margin:12px 0 6px;">YOUR QUESTION</div>'))
display(q_input)
display(widgets.HBox([ask_btn, clear_btn], layout=widgets.Layout(gap='8px', margin='8px 0')))
display(HTML('<div style="font-size:12px;font-weight:600;color:#6b7280;margin-bottom:6px;">CHAT</div>'))
display(out)


Textarea(value='', layout=Layout(height='60px', width='100%'), placeholder='Ask anything about your deliveries…

Output()